# 🎯 Auditor de Zona — Clasificador Binario Directo (RoBERTa Clínico)

En vez de clasificar las 7 categorías BI-RADS y agrupar después, este modelo se entrena **directamente** para la pregunta clínica binaria:

- **Zona segura (0):** BI-RADS 0, 1, 2, 3
- **Zona de riesgo (1):** BI-RADS 4, 5, 6

**Mismo modelo** (RoBERTa clínico), **mismos datos reales** (solo observaciones), **mismo split** (semilla 42, test 100% real). Lo único que cambia: la cabeza del modelo (2 salidas) y la etiqueta (2 zonas). Al juntar los pocos casos de riesgo en una sola clase, el modelo aprende **una** frontera en vez de tres imposibles.

**Métricas clave:**
- **AUC-ROC:** desempeño independiente del umbral (titular honesto y robusto).
- **Ajuste de umbral en validación** (no en test) para maximizar la detección de riesgo, evaluado luego en el test real.

## 1 · Configuración y librerías

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd, random, os, sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, f1_score
sys.path.append('..')
from src.preprocessing import MAX_LENGTH

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
print(f"Dispositivo: {DEVICE}")

Dispositivo: mps


## 2 · Datos: reagrupar a zona y dividir

Se divide con el **mismo split** que el auditor (estratificado por BI-RADS, semilla 42) y luego se reagrupa la etiqueta a zona. Así el test es idéntico al que evaluamos antes (11 riesgo / 643 segura), y la comparación es justa.

In [2]:
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
X = df['auditor_input'].values
y7 = df['BI-RADS'].values

# Mismo split que el auditor (sobre las 7 clases, semilla 42)
X_tv, X_test, y7_tv, y7_test = train_test_split(X, y7, test_size=0.15, random_state=SEED, stratify=y7)
X_train, X_val, y7_train, y7_val = train_test_split(X_tv, y7_tv, test_size=0.176, random_state=SEED, stratify=y7_tv)

# Reagrupar a zona: 0-3 -> 0 (segura), 4-6 -> 1 (riesgo)
z = lambda v: (np.asarray(v) >= 4).astype(int)
y_train, y_val, y_test = z(y7_train), z(y7_val), z(y7_test)

print(f"Train: {len(X_train)}  (riesgo: {y_train.sum()})")
print(f"Val:   {len(X_val)}  (riesgo: {y_val.sum()})")
print(f"Test:  {len(X_test)}  (riesgo: {y_test.sum()})")

Train: 3051  (riesgo: 51)
Val:   652  (riesgo: 11)
Test:  654  (riesgo: 11)


## 3 · Pesos de clase (binario)

In [3]:
pesos = compute_class_weight(class_weight='balanced', classes=np.array([0,1]), y=y_train)
peso_tensor = torch.tensor(pesos, dtype=torch.float32).to(DEVICE)
print(f"Peso zona segura: {pesos[0]:.2f} | Peso zona riesgo: {pesos[1]:.2f}")

Peso zona segura: 0.51 | Peso zona riesgo: 29.91


## 4 · Tokenizador, Dataset y DataLoader

In [4]:
try: tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; tokenizer=AutoTokenizer.from_pretrained(MODELO)

class ZonaDataset(Dataset):
    def __init__(self, t, l, tok, ml=MAX_LENGTH): self.t=list(t); self.l=list(l); self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

BATCH=16
train_loader=DataLoader(ZonaDataset(X_train,y_train,tokenizer),batch_size=BATCH,shuffle=True)
val_loader  =DataLoader(ZonaDataset(X_val,y_val,tokenizer),batch_size=BATCH)
test_loader =DataLoader(ZonaDataset(X_test,y_test,tokenizer),batch_size=BATCH)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

Batches -> train: 191 | val: 41 | test: 41


## 5 · Modelo binario (RoBERTa clínico + cabeza de 2 salidas)

In [5]:
class ZonaRoBERTa(nn.Module):
    def __init__(self, modelo, dropout=0.3):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(modelo)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(self.encoder.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        out=self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(out.last_hidden_state[:,0,:]))

model=ZonaRoBERTa(MODELO).to(DEVICE)
LR=3e-5; EPOCHS=12; PATIENCE=4
criterio=nn.CrossEntropyLoss(weight=peso_tensor)
optimizer=torch.optim.AdamW(model.parameters(), lr=LR)
scheduler=get_linear_schedule_with_warmup(optimizer,0,len(train_loader)*EPOCHS)
print("Modelo binario de zona listo (RoBERTa clínico).")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo binario de zona listo (RoBERTa clínico).


## 6 · Entrenamiento (selección del mejor modelo por AUC de validación)

In [6]:
def probs_y_labels(loader):
    model.eval(); ps, ls = [], []
    with torch.no_grad():
        for b in loader:
            ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE)
            p=F.softmax(model(ids,mask),dim=1)[:,1].cpu().numpy()  # prob de riesgo
            ps.extend(p); ls.extend(b['labels'].numpy())
    return np.array(ps), np.array(ls)

os.makedirs('../results',exist_ok=True); RUTA='../results/mejor_zona_binario.pt'
print("🏋️  Entrenando clasificador binario de zona")
print("Época | Train Loss | Val AUC")
mejor_auc, sin = 0.0, 0
for ep in range(1, EPOCHS+1):
    model.train(); tot=0.0
    for b in train_loader:
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lab=b['labels'].to(DEVICE)
        optimizer.zero_grad(); loss=criterio(model(ids,mask), lab)
        loss.backward(); optimizer.step(); scheduler.step(); tot+=loss.item()
    pv, lv = probs_y_labels(val_loader)
    auc = roc_auc_score(lv, pv)
    m=""
    if auc>mejor_auc: mejor_auc,sin=auc,0; torch.save(model.state_dict(),RUTA); m="  <- mejor"
    else: sin+=1
    print(f"  {ep:2d}  |  {tot/len(train_loader):.4f}  |  {auc:.4f}{m}")
    if sin>=PATIENCE: print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val AUC = {mejor_auc:.4f}")

🏋️  Entrenando clasificador binario de zona
Época | Train Loss | Val AUC
   1  |  0.4069  |  0.9000  <- mejor
   2  |  0.2584  |  0.9091  <- mejor
   3  |  0.2042  |  0.9319  <- mejor
   4  |  0.1418  |  0.9464  <- mejor
   5  |  0.1073  |  0.9308
   6  |  0.0803  |  0.9638  <- mejor
   7  |  0.0357  |  0.9230
   8  |  0.0281  |  0.9194
   9  |  0.0138  |  0.9011
  10  |  0.0123  |  0.9321
Early stopping en época 10
🏆 Mejor Val AUC = 0.9638


## 7 · AUC-ROC en test (métrica principal, independiente del umbral)

In [7]:
model.load_state_dict(torch.load(RUTA,map_location=DEVICE))
p_test, l_test = probs_y_labels(test_loader)
auc_test = roc_auc_score(l_test, p_test)
print("="*55)
print(f"  AUC-ROC (test): {auc_test:.4f}")
print("="*55)
print("Interpretación: probabilidad de que el modelo asigne mayor")
print("score de riesgo a un informe de riesgo que a uno seguro.")

  AUC-ROC (test): 0.9270
Interpretación: probabilidad de que el modelo asigne mayor
score de riesgo a un informe de riesgo que a uno seguro.


## 8 · Ajuste de umbral en validación y evaluación en test

El umbral se elige en **validación** (criterio de Youden: maximiza sensibilidad + especificidad) y luego se aplica al **test** (no se ajusta sobre el test, para no inflar). Se compara contra el umbral por defecto (0,5).

In [8]:
def metricas(y_true, y_prob, thr):
    y_pred=(y_prob>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    vpp=tp/(tp+fp) if (tp+fp) else 0
    return dict(thr=thr,sens=sens,espec=espec,vpp=vpp,tp=tp,fn=fn,fp=fp,tn=tn)

# Umbral de Youden sobre VALIDACIÓN
pv, lv = probs_y_labels(val_loader)
fpr, tpr, thrs = roc_curve(lv, pv)
youden = tpr - fpr
thr_opt = thrs[np.argmax(youden)]
thr_opt = float(np.clip(thr_opt, 0.01, 0.99))

print(f"Umbral óptimo (Youden, elegido en validación): {thr_opt:.3f}\n")
for nombre, thr in [("Umbral por defecto (0.50)", 0.5), (f"Umbral ajustado ({thr_opt:.3f})", thr_opt)]:
    m=metricas(l_test, p_test, thr)
    print(f"{nombre}:")
    print(f"   Sensibilidad (riesgo): {m['sens']:.4f}  |  Especificidad: {m['espec']:.4f}  |  VPP: {m['vpp']:.4f}")
    print(f"   Riesgo detectado: {m['tp']}/{m['tp']+m['fn']}  |  No detectados: {m['fn']}  |  Falsas alarmas: {m['fp']}")
    print()

Umbral óptimo (Youden, elegido en validación): 0.426

Umbral por defecto (0.50):
   Sensibilidad (riesgo): 0.6364  |  Especificidad: 0.9658  |  VPP: 0.2414
   Riesgo detectado: 7/11  |  No detectados: 4  |  Falsas alarmas: 22

Umbral ajustado (0.426):
   Sensibilidad (riesgo): 0.6364  |  Especificidad: 0.9642  |  VPP: 0.2333
   Riesgo detectado: 7/11  |  No detectados: 4  |  Falsas alarmas: 23



## 9 · Comparación con el enfoque anterior (7 clases agrupadas)

In [9]:
print("DETECCIÓN DE ZONA DE RIESGO — comparación (test real):")
print(f"   7 clases agrupadas a posteriori:   Sensibilidad 0.5455 | Especificidad 0.9938 | (sin AUC)")
m_def=metricas(l_test,p_test,0.5); m_opt=metricas(l_test,p_test,thr_opt)
print(f"   Binario directo (umbral 0.50):     Sensibilidad {m_def['sens']:.4f} | Especificidad {m_def['espec']:.4f}")
print(f"   Binario directo (umbral ajustado): Sensibilidad {m_opt['sens']:.4f} | Especificidad {m_opt['espec']:.4f}")
print(f"   AUC-ROC (binario):                 {auc_test:.4f}  <- métrica robusta principal")

DETECCIÓN DE ZONA DE RIESGO — comparación (test real):
   7 clases agrupadas a posteriori:   Sensibilidad 0.5455 | Especificidad 0.9938 | (sin AUC)
   Binario directo (umbral 0.50):     Sensibilidad 0.6364 | Especificidad 0.9658
   Binario directo (umbral ajustado): Sensibilidad 0.6364 | Especificidad 0.9642
   AUC-ROC (binario):                 0.9270  <- métrica robusta principal


## 10 · Cómo reportarlo (honestidad)

- **Titular:** AUC-ROC para detección de zona de riesgo (métrica robusta, independiente del umbral).
- **Punto de operación:** reportar el umbral ajustado y su sensibilidad/especificidad, dejando claro que el umbral se eligió en validación.
- **Para una red de seguridad clínica**, priorizar **sensibilidad** (detectar el máximo de casos de riesgo) es la elección correcta, asumiendo más falsas alarmas. Es una decisión clínica explícita, no un truco.
- **Limitación honesta ineludible:** solo 11 casos de riesgo reales en el test. Todas estas métricas tienen intervalos de confianza amplios. Es una **prueba de concepto**; la validación definitiva requiere más casos reales de categorías de riesgo (idealmente prospectivos).